In [1]:
#Q1. NLP Preprocessing Pipeline
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import wordpunct_tokenize

# These common words are used only if the NLTK dataset is not installed.
# For the full stopword list, run: python -m nltk.downloader stopwords
FALLBACK_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "by", "for",
    "from", "had", "has", "have", "he", "her", "his", "i", "in", "is",
    "it", "its", "of", "on", "or", "our", "she", "that", "the", "their",
    "them", "there", "these", "they", "this", "to", "was", "were",
    "will", "with", "you", "your",
}


def preprocess(sentence: str):
    """Return (original_tokens, tokens_without_stopwords, stemmed_words)."""
    # wordpunct_tokenize splits off punctuation without requiring punkt data.
    original_tokens = wordpunct_tokenize(sentence)
    try:
        english_stopwords = set(stopwords.words("english"))
    except LookupError:
        # Keep the exercise runnable in an offline Python environment.
        english_stopwords = FALLBACK_STOPWORDS

    # Punctuation is kept in the original-token list, but not the word lists.
    content_words = [
        token for token in original_tokens
        if token.isalpha() and token.lower() not in english_stopwords
    ]
    stemmer = PorterStemmer()
    stems = [stemmer.stem(word.lower()) for word in content_words]

    print("Original Tokens:", original_tokens)
    print("Tokens Without Stopwords:", content_words)
    print("Stemmed Words:", stems)
    return original_tokens, content_words, stems


if __name__ == "__main__":
    sentence = "NLP techniques are used in virtual assistants like Alexa and Siri."
    preprocess(sentence)


Original Tokens: ['NLP', 'techniques', 'are', 'used', 'in', 'virtual', 'assistants', 'like', 'Alexa', 'and', 'Siri', '.']
Tokens Without Stopwords: ['NLP', 'techniques', 'used', 'virtual', 'assistants', 'like', 'Alexa', 'Siri']
Stemmed Words: ['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri']


In [2]:
#Q2. Named Entity Recognition with SpaCy
import spacy


def show_named_entities(sentence: str):
    """Print each detected entity, label, and [start, end) character offsets."""
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError as exc:
        raise SystemExit(
            "spaCy's English model is required. Install it with: "
            "python -m spacy download en_core_web_sm"
        ) from exc

    doc = nlp(sentence)
    for entity in doc.ents:
        # spaCy's start_char is inclusive; end_char is exclusive.
        print(
            f"{entity.text} | {entity.label_} | "
            f"start={entity.start_char} | end={entity.end_char}"
        )
    return list(doc.ents)


if __name__ == "__main__":
    # The leading space is part of the original assigned input.
    sentence = (
        " LeBron James as the NBA player of the United States "
        "in Laker and won the NBA Final MVP in 2020."
    )
    show_named_entities(sentence)

LeBron James | PERSON | start=1 | end=13
NBA | ORG | start=21 | end=24
the United States | GPE | start=35 | end=52
Laker | GPE | start=56 | end=61
MVP | ORG | start=84 | end=87
2020 | DATE | start=91 | end=95


In [3]:
#Q3. Scaled Dot-Product Attention

import numpy as np


def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray):
    """Compute softmax((Q @ K.T) / sqrt(d_k)) @ V, and return both arrays."""
    d_k = K.shape[-1]  # Dimension of each key vector: 4 in this task.
    scores = (Q @ K.T) / np.sqrt(d_k)

    # Row-wise softmax; subtracting each maximum avoids numerical overflow.
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)

    output = attention_weights @ V
    return attention_weights, output


if __name__ == "__main__":
    Q = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
    K = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
    V = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])

    weights, attention_output = scaled_dot_product_attention(Q, K, V)
    print("Attention weights matrix:")
    print(np.round(weights, 3))
    print("Final output matrix:")
    print(np.round(attention_output, 3))

Attention weights matrix:
[[0.731 0.269]
 [0.269 0.731]]
Final output matrix:
[[2.076 3.076 4.076 5.076]
 [3.924 4.924 5.924 6.924]]
